In [1]:
# ============================================================
# FARM AI AGENT
# COMPLETE XGBOOST TRAINING PIPELINE
# DATASET 1: Indian Crop Production and Yield
# DATASET 2: Crop Yield with Soil and Weather
# DATASET 3: Crop Recommendation
# ============================================================

import os
import re
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

RANDOM_STATE = 42

# This notebook is expected to run from:
# RAG/ml_training/
#
# If the current working directory is different, the code
# automatically searches common project locations.

CURRENT_DIR = Path.cwd()

POSSIBLE_PROJECT_DIRS = [
    CURRENT_DIR,
    CURRENT_DIR / "ml_training",
    CURRENT_DIR.parent / "ml_training",
    Path("/mnt/d/EdgeAi_resistor_Dl_model/RAG/ml_training")
]

PROJECT_DIR = None

for possible_dir in POSSIBLE_PROJECT_DIRS:
    if (possible_dir / "data").exists():
        PROJECT_DIR = possible_dir
        break

if PROJECT_DIR is None:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PROJECT DIRECTORY")
print("=" * 80)
print(PROJECT_DIR)

print("\nDATA DIRECTORY")
print(DATA_DIR)

print("\nMODEL DIRECTORY")
print(MODEL_DIR)

print("\nOUTPUT DIRECTORY")
print(OUTPUT_DIR)

# ============================================================
# COMMON FUNCTIONS
# ============================================================

def clean_column_names(df):
    """
    Convert column names into consistent lowercase names.
    Example:
        Crop Name -> crop_name
        Crop_Year -> crop_year
    """

    df = df.copy()

    cleaned_columns = []

    for column in df.columns:
        column = str(column).strip().lower()
        column = re.sub(r"[^a-z0-9]+", "_", column)
        column = column.strip("_")
        cleaned_columns.append(column)

    df.columns = cleaned_columns

    return df


def load_dataset(filename):
    """
    Load a CSV file from the data directory.
    """

    file_path = DATA_DIR / filename

    if not file_path.exists():
        raise FileNotFoundError(
            f"\nDataset not found:\n{file_path}\n"
            f"\nAvailable files:\n"
            f"{list(DATA_DIR.iterdir()) if DATA_DIR.exists() else 'Data directory does not exist'}"
        )

    df = pd.read_csv(file_path)

    df = clean_column_names(df)

    print("\n" + "=" * 80)
    print("LOADED DATASET")
    print("=" * 80)
    print("File:", file_path)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

    return df


def clean_text_columns(df):
    """
    Clean object/string columns.
    """

    df = df.copy()

    for column in df.select_dtypes(include=["object"]).columns:
        df[column] = (
            df[column]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        df[column] = df[column].replace(
            {
                "nan": np.nan,
                "none": np.nan,
                "null": np.nan,
                "": np.nan,
                "na": np.nan,
                "n/a": np.nan
            }
        )

    return df


def convert_numeric_columns(df, columns):
    """
    Convert selected columns into numeric values.
    Invalid values become NaN.
    """

    df = df.copy()

    for column in columns:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )

    return df


def remove_duplicate_rows(df):
    """
    Remove duplicate rows.
    """

    before = len(df)

    df = df.drop_duplicates()
    df = df.reset_index(drop=True)

    after = len(df)

    print("Removed duplicate rows:", before - after)

    return df


def remove_invalid_values(df):
    """
    Replace infinity values with NaN.
    Remove impossible negative values from common
    agricultural numerical columns.
    """

    df = df.copy()

    df = df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    non_negative_columns = [
        "area",
        "production",
        "yield",
        "rainfall",
        "humidity",
        "nitrogen",
        "phosphorus",
        "potassium",
        "soil_moisture",
        "precipitation"
    ]

    for column in non_negative_columns:
        if column in df.columns:
            df.loc[df[column] < 0, column] = np.nan

    return df


def save_json(data, filename):
    """
    Save dictionary as JSON.
    """

    output_path = OUTPUT_DIR / filename

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            indent=4,
            default=str
        )

    print("Saved:", output_path)


def save_feature_importance(
    trained_pipeline,
    output_filename,
    plot_filename
):
    """
    Save feature importance from a pipeline.

    This correctly extracts the transformed feature names
    after OneHotEncoding.
    """

    try:
        preprocessor = trained_pipeline.named_steps["preprocessor"]
        model = trained_pipeline.named_steps["model"]

        feature_names = preprocessor.get_feature_names_out()

        importances = model.feature_importances_

        importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance": importances
        })

        importance_df = importance_df.sort_values(
            by="importance",
            ascending=False
        )

        csv_path = OUTPUT_DIR / output_filename

        importance_df.to_csv(
            csv_path,
            index=False
        )

        print("Saved:", csv_path)

        # Plot top 20 features
        top_features = importance_df.head(20)
        top_features = top_features.sort_values(
            by="importance",
            ascending=True
        )

        plt.figure(figsize=(10, 7))

        plt.barh(
            top_features["feature"],
            top_features["importance"]
        )

        plt.title("Top Feature Importance")
        plt.xlabel("Importance")
        plt.ylabel("Feature")

        plt.tight_layout()

        plot_path = OUTPUT_DIR / plot_filename

        plt.savefig(
            plot_path,
            dpi=150
        )

        plt.close()

        print("Saved:", plot_path)

        return importance_df

    except Exception as error:
        print("Feature importance could not be saved:", error)
        return None


def evaluate_regression_model(
    model,
    X_test,
    y_test,
    model_name
):
    """
    Evaluate regression model.
    """

    predictions = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)

    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")

    metrics = {
        "model": model_name,
        "task": "regression",
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
        "test_samples": int(len(y_test))
    }

    return metrics, predictions


def evaluate_classification_model(
    model,
    X_test,
    y_test,
    label_encoder,
    model_name
):
    """
    Evaluate classification model.
    """

    predictions_encoded = model.predict(X_test)

    predictions_encoded = np.asarray(
        predictions_encoded,
        dtype=int
    )

    predictions = label_encoder.inverse_transform(
        predictions_encoded
    )

    actual_labels = label_encoder.inverse_transform(
        np.asarray(y_test, dtype=int)
    )

    accuracy = accuracy_score(
        actual_labels,
        predictions
    )

    report = classification_report(
        actual_labels,
        predictions,
        output_dict=True,
        zero_division=0
    )

    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)

    print(f"Accuracy: {accuracy:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            actual_labels,
            predictions,
            zero_division=0
        )
    )

    metrics = {
        "model": model_name,
        "task": "classification",
        "accuracy": float(accuracy),
        "classification_report": report,
        "test_samples": int(len(y_test))
    }

    return metrics, predictions


# ============================================================
# DATASET 1
# INDIAN AGRICULTURE PRODUCTION AND YIELD
# ============================================================

def train_agriculture_production_model():

    print("\n\n")
    print("#" * 80)
    print("DATASET 1: INDIAN AGRICULTURE PRODUCTION AND YIELD")
    print("#" * 80)

    # Correct filename from your data folder
    df = load_dataset(
        "Indian_crop_production_yield_dataset.csv"
    )

    df = remove_duplicate_rows(df)
    df = clean_text_columns(df)

    # Rename possible column variations
    rename_map = {}

    for column in df.columns:

        if column in [
            "state_name",
            "state"
        ]:
            rename_map[column] = "state"

        elif column in [
            "district_name",
            "district"
        ]:
            rename_map[column] = "district"

        elif column in [
            "crop_name",
            "crop"
        ]:
            rename_map[column] = "crop"

        elif column in [
            "season_name",
            "season"
        ]:
            rename_map[column] = "season"

        elif column in [
            "crop_year",
            "year"
        ]:
            rename_map[column] = "year"

        elif column in [
            "area",
            "area_hectares"
        ]:
            rename_map[column] = "area"

        elif column in [
            "production",
            "production_tonnes"
        ]:
            rename_map[column] = "production"

        elif column in [
            "yield",
            "yield_tonnes_per_hectare"
        ]:
            rename_map[column] = "yield"

    df = df.rename(columns=rename_map)

    required_columns = [
        "state",
        "district",
        "crop",
        "season",
        "year",
        "area",
        "production",
        "yield"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"\nDataset 1 missing columns: {missing_columns}\n"
            f"Available columns: {list(df.columns)}"
        )

    df = convert_numeric_columns(
        df,
        [
            "year",
            "area",
            "production",
            "yield"
        ]
    )

    df = remove_invalid_values(df)

    df = df.dropna(
        subset=required_columns
    )

    # Physically valid agricultural values
    df = df[
        (df["area"] > 0) &
        (df["production"] >= 0) &
        (df["yield"] >= 0)
    ]

    # Feature engineering
    df["production_per_area"] = (
        df["production"] /
        df["area"].replace(0, np.nan)
    )

    df["year_sin"] = np.sin(
        2 * np.pi * df["year"] / 10
    )

    df["year_cos"] = np.cos(
        2 * np.pi * df["year"] / 10
    )

    feature_columns = [
        "state",
        "district",
        "crop",
        "season",
        "year",
        "area",
        "production",
        "year_sin",
        "year_cos"
    ]

    target_column = "yield"

    X = df[feature_columns]
    y = df[target_column]

    categorical_columns = [
        "state",
        "district",
        "crop",
        "season"
    ]

    numerical_columns = [
        "year",
        "area",
        "production",
        "year_sin",
        "year_cos"
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="median"
                            )
                        )
                    ]
                ),
                numerical_columns
            ),
            (
                "categorical",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="most_frequent"
                            )
                        ),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore"
                            )
                        )
                    ]
                ),
                categorical_columns
            )
        ]
    )

    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=600,
        learning_rate=0.04,
        max_depth=7,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.90,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                model
            )
        ]
    )

    # Time-aware split
    df = df.sort_values(
        by="year"
    )

    X = df[feature_columns]
    y = df[target_column]

    split_index = int(
        len(df) * 0.80
    )

    X_train = X.iloc[:split_index]
    X_test = X.iloc[split_index:]

    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]

    print("\nTraining Dataset 1...")

    pipeline.fit(
        X_train,
        y_train
    )

    metrics, predictions = evaluate_regression_model(
        pipeline,
        X_test,
        y_test,
        "Indian Agriculture Production and Yield Model"
    )

    save_json(
        metrics,
        "agriculture_production_metrics.json"
    )

    model_path = MODEL_DIR / "agriculture_production_xgb.pkl"

    joblib.dump(
        pipeline,
        model_path
    )

    print("Saved:", model_path)

    prediction_df = X_test.copy()

    prediction_df["actual_yield"] = y_test.values
    prediction_df["predicted_yield"] = predictions

    prediction_path = OUTPUT_DIR / (
        "agriculture_production_predictions.csv"
    )

    prediction_df.to_csv(
        prediction_path,
        index=False
    )

    print("Saved:", prediction_path)

    save_feature_importance(
        pipeline,
        "agriculture_production_feature_importance.csv",
        "agriculture_production_feature_importance.png"
    )

    return pipeline


# ============================================================
# DATASET 2
# CROP YIELD WITH SOIL AND WEATHER
# ============================================================

def train_soil_weather_yield_model():

    print("\n\n")
    print("#" * 80)
    print("DATASET 2: CROP YIELD WITH SOIL AND WEATHER")
    print("#" * 80)

    # Correct filename from your data folder
    df = load_dataset(
        "crop_yield.csv"
    )

    df = remove_duplicate_rows(df)
    df = clean_text_columns(df)

    rename_map = {}

    for column in df.columns:

        if column in [
            "crop_name",
            "crop"
        ]:
            rename_map[column] = "crop"

        elif column in [
            "soil_n",
            "nitrogen",
            "n"
        ]:
            rename_map[column] = "nitrogen"

        elif column in [
            "soil_p",
            "phosphorus",
            "p"
        ]:
            rename_map[column] = "phosphorus"

        elif column in [
            "soil_k",
            "potassium",
            "k"
        ]:
            rename_map[column] = "potassium"

        elif column in [
            "soil_ph",
            "ph"
        ]:
            rename_map[column] = "ph"

        elif column in [
            "rainfall",
            "rain"
        ]:
            rename_map[column] = "rainfall"

        elif column in [
            "temperature",
            "temp"
        ]:
            rename_map[column] = "temperature"

        elif column in [
            "humidity",
            "relative_humidity"
        ]:
            rename_map[column] = "humidity"

        elif column in [
            "yield",
            "crop_yield",
            "production_yield"
        ]:
            rename_map[column] = "yield"

    df = df.rename(columns=rename_map)

    required_columns = [
        "crop",
        "nitrogen",
        "phosphorus",
        "potassium",
        "ph",
        "rainfall",
        "temperature",
        "humidity",
        "yield"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"\nDataset 2 missing columns: {missing_columns}\n"
            f"Available columns: {list(df.columns)}"
        )

    numeric_columns = [
        "nitrogen",
        "phosphorus",
        "potassium",
        "ph",
        "rainfall",
        "temperature",
        "humidity",
        "yield"
    ]

    df = convert_numeric_columns(
        df,
        numeric_columns
    )

    df = remove_invalid_values(df)

    df = df.dropna(
        subset=required_columns
    )

    df = df[
        (df["nitrogen"] >= 0) &
        (df["phosphorus"] >= 0) &
        (df["potassium"] >= 0) &
        (df["ph"] > 0) &
        (df["rainfall"] >= 0) &
        (df["humidity"] >= 0) &
        (df["yield"] >= 0)
    ]

    # Feature engineering
    df["npk_total"] = (
        df["nitrogen"] +
        df["phosphorus"] +
        df["potassium"]
    )

    df["npk_ratio"] = (
        df["nitrogen"] /
        (
            df["phosphorus"] +
            df["potassium"] +
            1
        )
    )

    df["temperature_humidity_index"] = (
        df["temperature"] *
        df["humidity"] /
        100
    )

    df["rainfall_temperature_ratio"] = (
        df["rainfall"] /
        (
            abs(df["temperature"]) +
            1
        )
    )

    feature_columns = [
        "crop",
        "nitrogen",
        "phosphorus",
        "potassium",
        "ph",
        "rainfall",
        "temperature",
        "humidity",
        "npk_total",
        "npk_ratio",
        "temperature_humidity_index",
        "rainfall_temperature_ratio"
    ]

    target_column = "yield"

    X = df[feature_columns]
    y = df[target_column]

    categorical_columns = [
        "crop"
    ]

    numerical_columns = [
        "nitrogen",
        "phosphorus",
        "potassium",
        "ph",
        "rainfall",
        "temperature",
        "humidity",
        "npk_total",
        "npk_ratio",
        "temperature_humidity_index",
        "rainfall_temperature_ratio"
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="median"
                            )
                        )
                    ]
                ),
                numerical_columns
            ),
            (
                "categorical",
                Pipeline(
                    steps=[
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="most_frequent"
                            )
                        ),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore"
                            )
                        )
                    ]
                ),
                categorical_columns
            )
        ]
    )

    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=600,
        learning_rate=0.04,
        max_depth=7,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.90,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                model
            )
        ]
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE
    )

    print("\nTraining Dataset 2...")

    pipeline.fit(
        X_train,
        y_train
    )

    metrics, predictions = evaluate_regression_model(
        pipeline,
        X_test,
        y_test,
        "Soil and Weather Crop Yield Model"
    )

    save_json(
        metrics,
        "soil_weather_yield_metrics.json"
    )

    model_path = MODEL_DIR / "soil_weather_yield_xgb.pkl"

    joblib.dump(
        pipeline,
        model_path
    )

    print("Saved:", model_path)

    prediction_df = X_test.copy()

    prediction_df["actual_yield"] = y_test.values
    prediction_df["predicted_yield"] = predictions

    prediction_path = OUTPUT_DIR / (
        "soil_weather_yield_predictions.csv"
    )

    prediction_df.to_csv(
        prediction_path,
        index=False
    )

    print("Saved:", prediction_path)

    save_feature_importance(
        pipeline,
        "soil_weather_yield_feature_importance.csv",
        "soil_weather_yield_feature_importance.png"
    )

    return pipeline


# ============================================================
# DATASET 3
# CROP RECOMMENDATION DATASET
# ============================================================

def train_crop_recommendation_model():

    print("\n\n")
    print("#" * 80)
    print("DATASET 3: CROP RECOMMENDATION DATASET")
    print("#" * 80)

    # Correct filename from your data folder
    df = load_dataset(
        "Crop_recommendation.csv"
    )

    df = remove_duplicate_rows(df)
    df = clean_text_columns(df)

    rename_map = {}

    for column in df.columns:

        if column in [
            "n",
            "nitrogen",
            "soil_n"
        ]:
            rename_map[column] = "nitrogen"

        elif column in [
            "p",
            "phosphorus",
            "soil_p"
        ]:
            rename_map[column] = "phosphorus"

        elif column in [
            "k",
            "potassium",
            "soil_k"
        ]:
            rename_map[column] = "potassium"

        elif column in [
            "temperature",
            "temp"
        ]:
            rename_map[column] = "temperature"

        elif column in [
            "humidity",
            "relative_humidity"
        ]:
            rename_map[column] = "humidity"

        elif column in [
            "ph",
            "soil_ph"
        ]:
            rename_map[column] = "ph"

        elif column in [
            "rainfall",
            "rain"
        ]:
            rename_map[column] = "rainfall"

        elif column in [
            "label",
            "crop",
            "crop_name",
            "crop_label"
        ]:
            rename_map[column] = "crop"

    df = df.rename(columns=rename_map)

    required_columns = [
        "nitrogen",
        "phosphorus",
        "potassium",
        "temperature",
        "humidity",
        "ph",
        "rainfall",
        "crop"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"\nDataset 3 missing columns: {missing_columns}\n"
            f"Available columns: {list(df.columns)}"
        )

    numeric_columns = [
        "nitrogen",
        "phosphorus",
        "potassium",
        "temperature",
        "humidity",
        "ph",
        "rainfall"
    ]

    df = convert_numeric_columns(
        df,
        numeric_columns
    )

    df = remove_invalid_values(df)

    df = df.dropna(
        subset=required_columns
    )

    df = df[
        (df["nitrogen"] >= 0) &
        (df["phosphorus"] >= 0) &
        (df["potassium"] >= 0) &
        (df["temperature"] > -50) &
        (df["humidity"] >= 0) &
        (df["humidity"] <= 100) &
        (df["ph"] > 0) &
        (df["rainfall"] >= 0)
    ]

    # Feature engineering
    df["npk_total"] = (
        df["nitrogen"] +
        df["phosphorus"] +
        df["potassium"]
    )

    df["npk_ratio"] = (
        df["nitrogen"] /
        (
            df["phosphorus"] +
            df["potassium"] +
            1
        )
    )

    df["temperature_humidity_index"] = (
        df["temperature"] *
        df["humidity"] /
        100
    )

    feature_columns = [
        "nitrogen",
        "phosphorus",
        "potassium",
        "temperature",
        "humidity",
        "ph",
        "rainfall",
        "npk_total",
        "npk_ratio",
        "temperature_humidity_index"
    ]

    target_column = "crop"

    X = df[feature_columns]
    y_text = df[target_column]

    # This fixes the previous XGBoost error.
    # XGBoost requires numerical class labels.
    label_encoder = LabelEncoder()

    y = label_encoder.fit_transform(
        y_text
    )

    print("\nCrop classes:")
    print(
        list(label_encoder.classes_)
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y
    )

    model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        n_estimators=600,
        learning_rate=0.04,
        max_depth=7,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    print("\nTraining Dataset 3...")

    model.fit(
        X_train,
        y_train
    )

    metrics, predictions = evaluate_classification_model(
        model,
        X_test,
        y_test,
        label_encoder,
        "Crop Recommendation Model"
    )

    save_json(
        metrics,
        "crop_recommendation_metrics.json"
    )

    model_package = {
        "model": model,
        "label_encoder": label_encoder,
        "features": feature_columns
    }

    model_path = MODEL_DIR / "crop_recommendation_xgb.pkl"

    joblib.dump(
        model_package,
        model_path
    )

    print("Saved:", model_path)

    prediction_df = X_test.copy()

    prediction_df["actual_crop"] = (
        label_encoder.inverse_transform(
            y_test
        )
    )

    prediction_df["predicted_crop"] = predictions

    prediction_path = OUTPUT_DIR / (
        "crop_recommendation_predictions.csv"
    )

    prediction_df.to_csv(
        prediction_path,
        index=False
    )

    print("Saved:", prediction_path)

    # Confusion matrix
    predicted_encoded = model.predict(X_test)

    cm = confusion_matrix(
        y_test,
        predicted_encoded
    )

    plt.figure(figsize=(14, 11))

    plt.imshow(
        cm,
        interpolation="nearest"
    )

    plt.title(
        "Crop Recommendation Confusion Matrix"
    )

    plt.xlabel("Predicted Crop")
    plt.ylabel("Actual Crop")

    plt.colorbar()

    class_names = label_encoder.classes_

    tick_positions = np.arange(
        len(class_names)
    )

    plt.xticks(
        tick_positions,
        class_names,
        rotation=90
    )

    plt.yticks(
        tick_positions,
        class_names
    )

    plt.tight_layout()

    confusion_path = OUTPUT_DIR / (
        "crop_recommendation_confusion_matrix.png"
    )

    plt.savefig(
        confusion_path,
        dpi=150
    )

    plt.close()

    print("Saved:", confusion_path)

    # Feature importance
    importance_df = pd.DataFrame({
        "feature": feature_columns,
        "importance": model.feature_importances_
    })

    importance_df = importance_df.sort_values(
        by="importance",
        ascending=False
    )

    importance_path = OUTPUT_DIR / (
        "crop_recommendation_feature_importance.csv"
    )

    importance_df.to_csv(
        importance_path,
        index=False
    )

    print("Saved:", importance_path)

    plt.figure(figsize=(10, 7))

    top_features = importance_df.head(20)
    top_features = top_features.sort_values(
        by="importance",
        ascending=True
    )

    plt.barh(
        top_features["feature"],
        top_features["importance"]
    )

    plt.title(
        "Crop Recommendation Feature Importance"
    )

    plt.xlabel("Importance")
    plt.ylabel("Feature")

    plt.tight_layout()

    importance_plot_path = OUTPUT_DIR / (
        "crop_recommendation_feature_importance.png"
    )

    plt.savefig(
        importance_plot_path,
        dpi=150
    )

    plt.close()

    print("Saved:", importance_plot_path)

    return model_package


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def recommend_crops(
    nitrogen,
    phosphorus,
    potassium,
    temperature,
    humidity,
    ph,
    rainfall,
    top_n=5
):
    """
    Load the trained crop recommendation model
    and return the top recommended crops.
    """

    model_path = MODEL_DIR / (
        "crop_recommendation_xgb.pkl"
    )

    if not model_path.exists():
        raise FileNotFoundError(
            f"Model not found: {model_path}"
        )

    model_package = joblib.load(
        model_path
    )

    model = model_package["model"]
    label_encoder = model_package["label_encoder"]

    npk_total = (
        nitrogen +
        phosphorus +
        potassium
    )

    npk_ratio = (
        nitrogen /
        (
            phosphorus +
            potassium +
            1
        )
    )

    temperature_humidity_index = (
        temperature *
        humidity /
        100
    )

    input_data = pd.DataFrame([{
        "nitrogen": nitrogen,
        "phosphorus": phosphorus,
        "potassium": potassium,
        "temperature": temperature,
        "humidity": humidity,
        "ph": ph,
        "rainfall": rainfall,
        "npk_total": npk_total,
        "npk_ratio": npk_ratio,
        "temperature_humidity_index": temperature_humidity_index
    }])

    probabilities = model.predict_proba(
        input_data
    )[0]

    crop_names = label_encoder.inverse_transform(
        np.arange(
            len(probabilities)
        )
    )

    results = pd.DataFrame({
        "crop": crop_names,
        "probability": probabilities
    })

    results["probability_percent"] = (
        results["probability"] * 100
    )

    results = results.sort_values(
        by="probability",
        ascending=False
    )

    return results.head(top_n).reset_index(
        drop=True
    )


# ============================================================
# MAIN TRAINING PROGRAM
# ============================================================

print("\n")
print("=" * 80)
print("STARTING ALL XGBOOST TRAINING PIPELINES")
print("=" * 80)

training_results = {}

try:
    training_results["dataset_1"] = (
        train_agriculture_production_model()
    )

    print("\nDataset 1 completed successfully.")

except Exception as error:
    print("\nDataset 1 failed:")
    print(error)


try:
    training_results["dataset_2"] = (
        train_soil_weather_yield_model()
    )

    print("\nDataset 2 completed successfully.")

except Exception as error:
    print("\nDataset 2 failed:")
    print(error)


try:
    training_results["dataset_3"] = (
        train_crop_recommendation_model()
    )

    print("\nDataset 3 completed successfully.")

except Exception as error:
    print("\nDataset 3 failed:")
    print(error)


# ============================================================
# FINAL FILE VERIFICATION
# ============================================================

print("\n")
print("=" * 80)
print("TRAINING FINISHED")
print("=" * 80)

print("\nMODEL FILES:")

for file_path in sorted(MODEL_DIR.iterdir()):
    print(file_path)

print("\nOUTPUT FILES:")

for file_path in sorted(OUTPUT_DIR.iterdir()):
    print(file_path)

print("\n")
print("=" * 80)
print("FINAL FILE EXISTENCE CHECK")
print("=" * 80)

expected_models = [
    "agriculture_production_xgb.pkl",
    "soil_weather_yield_xgb.pkl",
    "crop_recommendation_xgb.pkl"
]

expected_outputs = [
    "agriculture_production_metrics.json",
    "agriculture_production_predictions.csv",
    "agriculture_production_feature_importance.csv",
    "soil_weather_yield_metrics.json",
    "soil_weather_yield_predictions.csv",
    "soil_weather_yield_feature_importance.csv",
    "crop_recommendation_metrics.json",
    "crop_recommendation_predictions.csv",
    "crop_recommendation_feature_importance.csv",
    "crop_recommendation_confusion_matrix.png"
]

print("\nExpected model files:")

for filename in expected_models:
    path = MODEL_DIR / filename
    print(
        filename,
        "->",
        "FOUND" if path.exists() else "MISSING"
    )

print("\nExpected output files:")

for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    print(
        filename,
        "->",
        "FOUND" if path.exists() else "MISSING"
    )

print("\nAll processing completed.")

PROJECT DIRECTORY
/mnt/d/EdgeAi_resistor_Dl_model/RAG/ml_training

DATA DIRECTORY
/mnt/d/EdgeAi_resistor_Dl_model/RAG/ml_training/data

MODEL DIRECTORY
/mnt/d/EdgeAi_resistor_Dl_model/RAG/ml_training/models

OUTPUT DIRECTORY
/mnt/d/EdgeAi_resistor_Dl_model/RAG/ml_training/outputs


STARTING ALL XGBOOST TRAINING PIPELINES



################################################################################
DATASET 1: INDIAN AGRICULTURE PRODUCTION AND YIELD
################################################################################

LOADED DATASET
File: /mnt/d/EdgeAi_resistor_Dl_model/RAG/ml_training/data/Indian_crop_production_yield_dataset.csv
Shape: (575879, 8)
Columns: ['state_name', 'district_name', 'crop_year', 'season', 'crop', 'area', 'production', 'yield']
Removed duplicate rows: 0

Training Dataset 1...

Indian Agriculture Production and Yield Model
MAE  : 1936.1038
RMSE : 32341.0443
R2   : 0.8788
Saved: /mnt/d/EdgeAi_resistor_Dl_model/RAG/ml_training/outputs/agriculture_pro